# ?? ??

### ??? ????

In [ ]:
# ??? ????
from sklearn.datasets import load_diabetes
import pandas as pd

diab = load_diabetes()
data = pd.DataFrame(data=diab.data, columns=diab.feature_names)

### ?? EDA

In [ ]:
# ?? ??? ????
# ???? ????
data.info()

In [ ]:
# ?? ? ?? ????
data.head()


In [ ]:
# ?? ??? ????
data.describe()

### DataFrame?? ???? ?? ???.?

**AttributeError** ??
- ????? ??(Object)? ???? ?? ??(attribute)?? ???(method)? ????? ? ? ???? ??
-  "? ???? ?? ?? ??? ??!"?? ???

---

EDA?? ??? info(), head(), describe()? ?? ??? Pandas? DataFrame??? ?? ??? **?? ??**

In [ ]:
diab.info()

In [ ]:
diab.head()

In [ ]:
diab.describe()

### ??? (??? ???)

**Bunch = ???? ?? ??**
- ?? ??? ???? ??? ??? ??(Container)
- ??? ??(data, target)?? ?? ?? ???? ???


---
*diab = load_diabetes()*

? ??? ??? ????? ??? ???

=> diab? Bunch ??? ???????.

---
**?? ? DataFrame?? ???? ????**

EDA?? ??? ?? (info, head, describe)?? DataFrame ?? ???? ????

?? ??,
- Pandas? DataFrame = ?? ??? ??
- sklearn? Bunch = ??? ??

?? ???, EDA ???? "?? ?? ?? ??????" ?? ??? ???

.........

DataFrame(??? ??)? ?? ???????   ?   ? ? ??? ?? ??? ?????

Bunch(??? ??)? ?? ???????   ? ?????? ?? ?? ??? ???? (??)

??? ??? ?????

**Bunch? ?? ???? ?? ????**

- .data() = ?? ??
- .target() = ?? ??
- feature_names() = ???


### ????? ???? ??

In [ ]:
# ????(????)? ????(????) ??
X = pd.DataFrame(diab.data, columns=diab.feature_names)
y = pd.Series(diab.target, name='target')

In [ ]:
# BMI? ?? ?? ??? ??
# ??, ??? ??? ??
from sklearn.model_selection import train_test_split

# bmi ??? ??
X_bmi = X[['bmi']]
X_train, X_test, y_train, y_test = train_test_split(X_bmi, y, random_state=42)

### (???!) ?? ??? ????

In [ ]:
# ?? ?? ?? ???? ? ?? ?????
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

In [ ]:
# y? ?????
# ?? ???? ??? ???
import matplotlib.pyplot as plt

y_pred = lr.predict(X_test)

plt.scatter(X_test, y_test, label='actual')
plt.plot(X_test, y_pred, color='red', label='predict line')
plt.xlabel('BMI')
plt.ylabel('target')
plt.legend()
plt.show()

print('Coef:', lr.coef_[0], 'Intercept:', lr.intercept_)

In [ ]:
# ?? ???? ??? ???
import numpy as np
a = lr.coef_[0]
b = lr.intercept_

x = np.linspace(-0.10,0.15)
y = a*x+b

plt.plot(x, y, color='red', linewidth=2, alpha=0.6, label='y=ax+b')
plt.scatter(X_test, y_pred, label='predicted')
plt.legend()
plt.show()

# ?? ???

In [ ]:
#????? ??
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:
#??? ????
df = pd.read_csv('kaggle_data.csv')
df.head(5)

In [ ]:
# 'date' ??? datetime ??? ??
df['date'] = pd.to_datetime(df['date'])

# ?? ?? ?? ??
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek # ???=0, ???=6
df['day_of_year'] = df['date'].dt.dayofyear
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int) # ?? ??
df['quarter'] = df['date'].dt.quarter

In [ ]:
#??? ????
def simple_summary(df):
    print("\n" + "="*50)
    print(f"       [ DATASET SUMMARY ]")
    print("="*50)
    print(f"? Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print("-"*50)
    print(f"{'Column Name':<20} | {'Dtype':<10} | {'Nulls'}")
    print("-"*50)

    for col in df.columns:
        null_count = df[col].isnull().sum()
        dtype = str(df[col].dtype)
        # ???? ??? ??(?) ??
        null_str = f"{null_count} (!!)" if null_count > 0 else f"{null_count}"
        print(f"{col[:20]:<20} | {dtype:<10} | {null_str}")

    print("="*50 + "\n")

# ?? ??
simple_summary(df)

In [ ]:
categorical_cols = ['country', 'store', 'product','date']

# 2. LabelEncoder ??
le = LabelEncoder()

for col in categorical_cols:
    # ? ???? ?? ? ??
    df[col] = le.fit_transform(df[col])
    print(f"? {col:<10} ?? ??")

print("\n" + "="*30)
print(df[categorical_cols].dtypes)

In [ ]:
df = df.dropna(subset=['num_sold'])

X=df.drop(["num_sold"],axis=1)
y=df["num_sold"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=33)

In [ ]:
rf = RandomForestRegressor(random_state=33, n_jobs=-1)

# 2. ??????? ??? (?? ??? ?? ??? ??)
param_grid = {
    'n_estimators': [100],
    'max_depth': [10, 15],
    'min_samples_split': [2, 4]
}

# 3. GridSearchCV ?? (MAPE ??)
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_percentage_error', # MAPE ??
    verbose=2
    #1? ???? 2? ????
)

# 4. ?? ??
print("?? MAPE ???? GridSearchCV ??? ??...")
grid_search.fit(X_train, y_train)

# 5. ?? ? ?? MAPE ??
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

# ?? MAPE (0.1?? 10% ??? ??)
final_mape = mean_absolute_percentage_error(y_test, y_pred)

print("\n" + "="*50)
print(f"? ?? ????: {grid_search.best_params_}")
print(f"?? ?? ?? MAPE: {final_mape:.4f} ({final_mape*100:.2f}%)")
print("="*50)